# 03 — Custom inputs: IAM files and inventories

**Audience:** Users bringing local IAM outputs or additional `bw2io`-compatible inventory workbooks into Premise.

**Prerequisites:** A configured Brightway project, a local IAM file, and optionally an inventory workbook compatible with the source ecoinvent version.

**Learning goals:** name and locate a custom IAM file, declare additional inventories, and build a scenario without embedding machine-specific paths.


## Outline

1. Configure local paths through environment variables.
2. Validate the expected IAM filename.
3. Declare optional inventory workbooks.
4. Build and update the scenario.


In [ ]:
import os
from pathlib import Path

import bw2data as bd

from premise import NewDatabase

PROJECT = "ecoinvent-3.12-cutoff"
SOURCE_DATABASE = "ecoinvent-3.12-cutoff"
BIOSPHERE_DATABASE = "ecoinvent-3.12-biosphere"
SOURCE_VERSION = "3.12"

iam_dir_value = os.environ.get("PREMISE_IAM_FILES_DIR")
if not iam_dir_value:
    raise RuntimeError("Set PREMISE_IAM_FILES_DIR to your local IAM directory.")
IAM_FILES_DIR = Path(iam_dir_value).expanduser()

bd.projects.set_current(PROJECT)


## 1. Name the custom IAM file

Premise looks for `{model}_{pathway}.csv`, `.mif`, or `.xlsx` in the directory supplied through the scenario's `filepath` field.


In [ ]:
MODEL = "remind"
PATHWAY = "my_special_scenario"
YEAR = 2030

candidates = [
    IAM_FILES_DIR / f"{MODEL}_{PATHWAY}{suffix}" for suffix in (".csv", ".mif", ".xlsx")
]
if not any(path.is_file() for path in candidates):
    raise FileNotFoundError(f"Expected one of: {candidates}")


## 2. Add optional inventories

Set `PREMISE_ADDITIONAL_INVENTORY` to a local Excel workbook. Its exchanges must include explicit names, products, locations, and types. Zero amounts should only be used when Premise is expected to fill them from documented replacement metadata.


In [ ]:
inventory_value = os.environ.get("PREMISE_ADDITIONAL_INVENTORY")
additional_inventories = []
if inventory_value:
    inventory_path = Path(inventory_value).expanduser()
    if not inventory_path.is_file():
        raise FileNotFoundError(inventory_path)
    additional_inventories.append(
        {
            "filepath": str(inventory_path),
            "ecoinvent version": SOURCE_VERSION,
        }
    )


## 3. Build and update

A local IAM file does not need an IAM download key. The chosen model must still have valid Premise mappings and a topology.


In [ ]:
scenario = {
    "model": MODEL,
    "pathway": PATHWAY,
    "year": YEAR,
    "filepath": str(IAM_FILES_DIR),
}

ndb = NewDatabase(
    scenarios=[scenario],
    source_db=SOURCE_DATABASE,
    source_version=SOURCE_VERSION,
    biosphere_name=BIOSPHERE_DATABASE,
    additional_inventories=additional_inventories or None,
)
ndb.update(["electricity"])


## Pitfalls and extension

- A custom pathway name does not replace the need for model-specific variable mappings.
- Inventory workbooks must target the declared ecoinvent version.
- Inspect written exchanges after export; do not assume zero-filled or replacement exchanges linked correctly.

## Exercise

Add a second inventory workbook while keeping the list construction deterministic.


In [ ]:
second_inventory = Path("path/to/second-inventory.xlsx")
exercise_entry = {
    "filepath": str(second_inventory),
    "ecoinvent version": SOURCE_VERSION,
}
